# Õppetund 08 - Mitmeagendi disainimuster


## Seadistamine


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Miks kasutada mitmeagendilisi süsteeme?

Reaalsete probleemide lahendamine, nagu reisiplaani koostamine, nõuab palju erinevaid teadmisi — logistikat, kohalikku teadmist, eelarvestamist ja muud. Üks agent, kes püüab kõike korraga hallata, muutub kiiresti ebapraktiliseks.

Mitmeagendilised süsteemid lahendavad seda **eristumise** kaudu: iga agent keskendub ühele teadmistevaldkonnale, tootes parema kvaliteediga tulemusi kui üldteadmistega agent. Need parandavad ka **skaalautuvust** — võite lisada uusi agente (nt lennuspetsialist, restoranikriitik) ilma olemasolevat töövoogu ümber kirjutamata. Agendid töötavad koos läbi struktureeritud torujuhtme, andes konteksti ühest teise.


## Spetsialiseeritud agentide loomine


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Jada tööskeemi loomine

`WorkflowBuilder` võimaldab sul ühendatud agendid suunatud graafikusse ühendada. Siin loome lihtsa kaheastmelise torujuhtme: **TravelPlanner** koostab marsruudi kavandi, seejärel vaatab ja täiustab seda **TravelConcierge**.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Rohkem agentide lisamine töövoogu

Üks suurimaid eeliseid mitmeagendi mustri puhul on selle lihtne laiendatavus. Allpool lisame **BudgetReviewer** agendi, kes kontrollib plaani reisija eelarve suhtes, märgistab üksused, mis võivad kulud üle piiri viia, ja pakub raha säästmise alternatiive. Töövoog käivitab nüüd kolm agenti järjest:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Kokkuvõte

Selles õppetükis õppisite, kuidas:

1. **Luuakse spetsialiseerunud agendid** — igaüks keskendunud kindlale ülesandele (planeerimine, konverentsitöötaja, eelarve ülevaatus).
2. **Ühendate agendid järjestikuseks töövooguks** kasutades `WorkflowBuilder` ja `add_edge`.
3. **Voogedastate väljundit** mitme agendiga torustikust, jälgides, milline agent räägib.
4. **Laiendate töövoogu** lisades uusi agente ahelasse ilma olemasolevaid muutmata.

Mitme agendi disainimuster hoiab iga agendi lihtsana, samal ajal tootes rikkalikumaid, põhjalikumalt üle vaadatud tulemusi kui ükski agent üksinda suudaks saavutada.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
